<a href="https://colab.research.google.com/github/NoT-Serna/Juan_Serna_FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content page for one client on one reporting date. Im using a one month time window of April 2026

In [17]:
import duckdb
from google.colab import userdata

# Read the token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Register the Hugging Face secret
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

# Warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Test the connection
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



In [19]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE DATE_TRUNC('month', report_date) = DATE '2026-04-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬────────────┬────────────┐
│   rows   │ start_date │  end_date  │
│  int64   │    date    │    date    │
├──────────┼────────────┼────────────┤
│ 10424730 │ 2026-04-01 │ 2026-04-30 │
└──────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:**
1.   gsc_impressions
2.   gsc_clicks
3.   sessions_ogranic
4.   ctr
5.   scroll_events
6.   content_type
7.   main_intent
8.   word_count


**Label:** I choose clasifiacion therefore my leakage-safe label will be
1. needs_review (created by me)

**Context**
1. content_hash_id
2. client_hash_id
3. report_date

**Excluded**
1. client_has_gsc: It indicates data availability not an SEO feature for CTR and Engagement
2. gsc_data_available: Another data indicator not an SEO feature



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1**
Claim: One row represents one content page for one client on one reporting date

In [26]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS c
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE DATE_TRUNC('month', report_date) = DATE '2026-04-01'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

**Query 2**
Claim: My analysis slice covers one month (Ej: January 2025)

In [21]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE DATE_TRUNC('month', report_date) = DATE '2026-04-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬────────────┬────────────┐
│   rows   │ start_date │  end_date  │
│  int64   │    date    │    date    │
├──────────┼────────────┼────────────┤
│ 10424730 │ 2026-04-01 │ 2026-04-30 │
└──────────┴────────────┴────────────┘

**Query 3** Claim: I will use rows where the requierd data is available (GSC data or GA4+ GSC data)



In [25]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE DATE_TRUNC('month', report_date) = DATE '2026-04-01'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE;
  """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         462315 │
└────────────────┘

**First Five Features**

1.  **gsc_impressions:** Measures search visibility
2.  **gsc_sum_position:** Search position is related to expected CTR
3.   **content_type:** Different content types may have different CTR patterns
4. **main_intnent:** Search intent affects click behavior
5. **word_count:** Content length may influence engagement and CTR



In [36]:
con.sql(f"""
SELECT
    f.report_date,
    f.client_hash_id,
    f.content_hash_id,

    -- Five features
    f.gsc_impressions,
    f.gsc_sum_position,
    d.content_type,
    d.main_intent,
    d.word_count

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') AS f

JOIN read_parquet('{rel}/dim_content.parquet') AS d
ON f.content_hash_id = d.content_hash_id

WHERE DATE_TRUNC('month', f.report_date) = DATE '2026-04-01'
  AND f.gsc_data_available IS TRUE

LIMIT 20;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬──────────────────┬─────────────────┬───────────────┬────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_sum_position │  content_type   │  main_intent  │ word_count │
│    date     │         varchar         │         varchar          │      int64      │      int64       │     varchar     │    varchar    │   int64    │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼──────────────────┼─────────────────┼───────────────┼────────────┤
│ 2026-04-30  │ client_06d356715a8ff3b6 │ content_0059a4d4195810c9 │              92 │              741 │ keyword article │ informational │       2087 │
│ 2026-04-29  │ client_06d356715a8ff3b6 │ content_005b6b7f7b8dda7f │              69 │              456 │ keyword article │ informational │       2567 │
│ 2026-04-29  │ client_06d356715a8ff3b6 │ content_0153b7dedc3fc40d │              

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data could never tell me information form the period Im trying to predcit. So the label needs_review cannot include features from the month Im trying to predict. Another thing to take into account is that rows that contain GSC data and GA4 data should be used for engagement metrics since rows wihtout GA4 data available is not enough for the needs_review label.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.